# BERT Crisis Classifier — trained on the shared train/val/test split (severity dropped)
0 = no_crisis, 1 = implicit_crisis, 2 = explicit_crisis

Run in Google Colab: Runtime > Change runtime type > GPU (T4 is fine)

**What changed vs. the previous version:**
- No more severity-to-3-class mapping — `train.csv` / `val.csv` / `test.csv` already contain a
  ready-made `label` column (0/1/2), so we just read it directly.
- `severity` column has been dropped from the CSVs (via the Copilot cleanup step) and is no
  longer referenced anywhere in this notebook.
- Everything else (class weighting, training loop, evaluation) is unchanged.


## 0. SETUP

In [1]:
# !pip install -q transformers datasets scikit-learn torch --upgrade

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from torch.utils.data import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


## 1. UPLOAD / LOAD THE SHARED TRAIN / VAL / TEST SPLITS
These files should already have `severity` dropped and contain: `row_index, url, content, label, class_name, split`.


In [2]:
from google.colab import files

print("Upload train.csv, val.csv, and test.csv (severity column already dropped)")
uploaded = files.upload()

# --- CONFIG: adjust these only if your teammate's files use different names ---
TRAIN_CSV = "train.csv"
VAL_CSV = "validation.csv"
TEST_CSV = "test.csv"

TEXT_COL = "content"   # column with the post text
LABEL_COL = "label"    # column with the ready-made 0/1/2 label
# -------------------------------------------------------------------------


Upload train.csv, val.csv, and test.csv (severity column already dropped)


Saving test.csv to test.csv
Saving train.csv to train.csv
Saving validation.csv to validation.csv


## 2. LOAD + VALIDATE (no severity mapping needed anymore)

In [3]:
label_names = ["no_crisis", "implicit_crisis", "explicit_crisis"]


def load_and_prepare(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
    # sanity check: label must already be 0/1/2
    bad = set(df[LABEL_COL].unique().tolist()) - {0, 1, 2}
    if bad:
        raise ValueError(
            f"Found label values outside {{0,1,2}} in {path}: {bad}. "
            "Check that this file really has the collapsed 'label' column, not raw severity."
        )
    return df


train_df = load_and_prepare(TRAIN_CSV)
val_df = load_and_prepare(VAL_CSV)
test_df = load_and_prepare(TEST_CSV)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"--- {name}: n={len(df)} ---")
    print(df[LABEL_COL].value_counts().sort_index())
    print()

train_texts, train_labels = train_df[TEXT_COL].astype(str).tolist(), train_df[LABEL_COL].tolist()
val_texts, val_labels = val_df[TEXT_COL].astype(str).tolist(), val_df[LABEL_COL].tolist()
test_texts, test_labels = test_df[TEXT_COL].astype(str).tolist(), test_df[LABEL_COL].tolist()


--- train: n=905 ---
label
0    245
1    227
2    433
Name: count, dtype: int64

--- val: n=194 ---
label
0    53
1    48
2    93
Name: count, dtype: int64

--- test: n=195 ---
label
0    53
1    49
2    93
Name: count, dtype: int64



## 3. TOKENIZATION / DATASET

In [4]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)


class CrisisDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length", max_length=max_len
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = CrisisDataset(train_texts, train_labels, tokenizer)
val_dataset = CrisisDataset(val_texts, val_labels, tokenizer)
test_dataset = CrisisDataset(test_texts, test_labels, tokenizer)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## 4. CLASS WEIGHTS

In [5]:
present_classes = np.unique(train_labels)
print("Classes present in train split:", present_classes)
print("Train label counts:", pd.Series(train_labels).value_counts().sort_index().to_dict())
missing = set([0, 1, 2]) - set(present_classes.tolist())
if missing:
    raise ValueError(f"Class(es) {missing} have zero examples in the training split.")

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=present_classes,
    y=np.array(train_labels),
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).to(device)
print("Class weights (no_crisis, implicit_crisis, explicit_crisis):", class_weights_np)


Classes present in train split: [0 1 2]
Train label counts: {0: 245, 1: 227, 2: 433}
Class weights (no_crisis, implicit_crisis, explicit_crisis): [1.23129252 1.32892805 0.69668976]


## 5. MODEL + WEIGHTED LOSS TRAINER

In [6]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. METRICS

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


## 7. TRAINING ARGS

In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=int(0.1 * (len(train_dataset) / 16) * 8),  # ~10% of total steps
    weight_decay=0.01,
    eval_strategy="epoch",       # older transformers: use evaluation_strategy="epoch"
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,  # keep ONLY the best checkpoint on disk
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


## 8. TRAIN

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,1.105524,1.017963,0.438144,0.634928,0.505216,0.432228
2,0.880162,0.680907,0.737113,0.741385,0.742941,0.728906
3,0.560752,0.644340,0.742268,0.757488,0.762016,0.737862
4,0.302598,0.574387,0.793814,0.785835,0.779747,0.780209
5,0.178988,0.627482,0.809278,0.804766,0.813366,0.801307
6,0.084671,0.723082,0.814433,0.814456,0.794739,0.797768
7,0.045419,0.707465,0.814433,0.806144,0.802200,0.803629
8,0.008157,0.732361,0.819588,0.809913,0.812504,0.810149


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=456, training_loss=0.34902426041662693, metrics={'train_runtime': 479.0949, 'train_samples_per_second': 15.112, 'train_steps_per_second': 0.952, 'total_flos': 952470572175360.0, 'train_loss': 0.34902426041662693, 'epoch': 8.0})

## 9. EVALUATE ON TEST SET

In [10]:
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=1)
test_true = test_results.label_ids

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)
print("Accuracy:", accuracy_score(test_true, test_preds))
print()
print(
    classification_report(
        test_true, test_preds, target_names=label_names, digits=4, zero_division=0
    )
)
print("Confusion matrix (rows = true, cols = predicted):")
print(
    pd.DataFrame(
        confusion_matrix(test_true, test_preds),
        index=[f"true_{n}" for n in label_names],
        columns=[f"pred_{n}" for n in label_names],
    )
)


FINAL TEST RESULTS
Accuracy: 0.7692307692307693

                 precision    recall  f1-score   support

      no_crisis     0.7660    0.6792    0.7200        53
implicit_crisis     0.6538    0.6939    0.6733        49
explicit_crisis     0.8333    0.8602    0.8466        93

       accuracy                         0.7692       195
      macro avg     0.7510    0.7444    0.7466       195
   weighted avg     0.7699    0.7692    0.7686       195

Confusion matrix (rows = true, cols = predicted):
                      pred_no_crisis  pred_implicit_crisis  \
true_no_crisis                    36                    12   
true_implicit_crisis               4                    34   
true_explicit_crisis               7                     6   

                      pred_explicit_crisis  
true_no_crisis                           5  
true_implicit_crisis                    11  
true_explicit_crisis                    80  


## 10. SAVE MODEL

In [11]:
# Save in fp16 to roughly halve file size for easier hosting/pushing
model.half()
model.save_pretrained("./best_model_fp16")
tokenizer.save_pretrained("./best_model_fp16")
print("Model saved to ./best_model_fp16")

# Optional: zip and download from Colab
# !zip -r best_model_fp16.zip ./best_model_fp16
# from google.colab import files
# files.download("best_model_fp16.zip")
# Optional: zip and download from Colab
!zip -r best_model_fp16.zip ./best_model_fp16
from google.colab import files
files.download("best_model_fp16.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./best_model_fp16


In [14]:
!zip -r best_model_fp16.zip ./best_model_fp16
from google.colab import files
files.download("best_model_fp16.zip")

  adding: best_model_fp16/ (stored 0%)
  adding: best_model_fp16/model.safetensors (deflated 8%)
  adding: best_model_fp16/tokenizer_config.json (deflated 43%)
  adding: best_model_fp16/tokenizer.json (deflated 71%)
  adding: best_model_fp16/config.json (deflated 54%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. PUSH TO HUGGING FACE HUB (recommended instead of committing the model to GitHub)
Uncomment and run once you have a token from huggingface.co/settings/tokens


In [12]:
# from huggingface_hub import HfApi, login
# login()  # paste your HF token when prompted
# api = HfApi()
# api.create_repo("your-username/suicide-risk-bert", exist_ok=True)
# model.push_to_hub("your-username/suicide-risk-bert")
# tokenizer.push_to_hub("your-username/suicide-risk-bert")
